In [13]:
from credit_risk.data.readers import read_performance,read_origination
from credit_risk.target.delinquency import (
    build_24m_serious_delinquency_target,
    build_cohort_eligibility,
    build_outcome_observability
)

In [7]:
perf = read_performance("data/01_raw/freddie_mac/2015/sample_perf_2015.txt")

In [8]:
from credit_risk.target.delinquency import (
    build_24m_serious_delinquency_target,
)

target = build_24m_serious_delinquency_target(perf)

print("Final cohort:", len(target))

print(target["ever_90dpd_24m"].value_counts().sort_index())

print(target["ever_90dpd_24m"].value_counts(normalize=True).sort_index())

Final cohort: 47255
ever_90dpd_24m
0    46912
1      343
Name: count, dtype: int64
ever_90dpd_24m
0    0.992742
1    0.007258
Name: proportion, dtype: float64


In [10]:
eligibility = build_cohort_eligibility(perf)
observability = build_outcome_observability(perf)

qc = eligibility.merge(
    observability,
    on="loan_id",
    how="left",
    validate="one_to_one",
)

excluded_after_start = qc.loc[
    qc["is_start_eligible"] & ~qc["is_outcome_observable"].fillna(False)
]

print("Start eligible:", qc["is_start_eligible"].sum())
print("Unobservable after start eligibility:", len(excluded_after_start))

excluded_after_start[
        [
            "loan_id",
            "first_loan_age",
            "last_loan_age",
            "ever_serious_delinquency",
            "final_zero_balance_code",
        ]
    ]


Start eligible: 47281
Unobservable after start eligibility: 26


,loan_id,first_loan_age,last_loan_age,ever_serious_delinquency,final_zero_balance_code
84,F15Q10002812,0,11.0,False,96
1625,F15Q10047986,1,8.0,False,96
3412,F15Q10102794,0,7.0,False,96
4608,F15Q10138012,0,7.0,False,96
4729,F15Q10141205,0,8.0,False,96
5756,F15Q10171273,0,4.0,False,96
10841,F15Q10319507,1,9.0,False,96
13940,F15Q20048207,0,4.0,False,96
15510,F15Q20098908,0,5.0,False,96
20729,F15Q20269413,1,8.0,False,96


In [16]:
orig = read_origination("data/01_raw/freddie_mac/2015/sample_orig_2015.txt")
print(orig.shape)
model_base = orig.merge(
    target,
    on="loan_id",
    how="inner",
    validate="one_to_one",
)

print(model_base.shape)

(50000, 31)
(47255, 32)


In [19]:
borrower_features = [
    "credit_score",
    "vantage_score_4",
    "original_dti",
    "number_of_borrowers",
    "first_time_homebuyer_flag",
]

for col in borrower_features:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print("dtype:", model_base[col].dtype)
    print("missing:", model_base[col].isna().sum())
    print("unique:", model_base[col].nunique(dropna=False))

    print("\nMost common values:")
    print(model_base[col].value_counts(dropna=False).head(15))


credit_score
dtype: int64
missing: 0
unique: 317

Most common values:
credit_score
801    615
787    551
790    547
797    533
791    532
796    526
802    506
776    488
809    487
800    485
798    478
799    467
788    465
778    464
793    461
Name: count, dtype: int64

vantage_score_4
dtype: int64
missing: 0
unique: 1

Most common values:
vantage_score_4
9999    47255
Name: count, dtype: int64

original_dti
dtype: int64
missing: 0
unique: 51

Most common values:
original_dti
999    3899
44     2030
43     1883
45     1806
41     1794
42     1773
39     1694
40     1693
38     1615
36     1583
37     1558
33     1484
34     1470
31     1411
32     1410
Name: count, dtype: int64

number_of_borrowers
dtype: int64
missing: 0
unique: 2

Most common values:
number_of_borrowers
2    24078
1    23177
Name: count, dtype: int64

first_time_homebuyer_flag
dtype: str
missing: 0
unique: 3

Most common values:
first_time_homebuyer_flag
N    39954
Y     7300
9        1
Name: count, dtype: int64

In [21]:
for col in [
    "credit_score",
    "original_dti",
    "number_of_borrowers",
]:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print(model_base[col].describe())

    print("\nLowest values:")
    print(model_base[col].value_counts().sort_index().head(15))

    print("\nHighest values:")
    print(model_base[col].value_counts().sort_index().tail(15))


credit_score
count    47255.000000
mean       749.152661
std         47.731212
min        462.000000
25%        717.000000
50%        759.000000
75%        788.000000
max        832.000000
Name: credit_score, dtype: float64

Lowest values:
credit_score
462    1
476    1
479    1
482    1
483    1
485    1
486    1
494    2
497    1
500    1
501    1
505    1
507    1
509    1
512    2
Name: count, dtype: int64

Highest values:
credit_score
816    216
817    134
818     52
819     90
820     81
821     10
822     15
823     63
824      3
825      7
826      3
827      2
829      5
831      1
832      2
Name: count, dtype: int64

original_dti
count    47255.000000
mean       113.518273
std        265.696033
min          1.000000
25%         28.000000
50%         36.000000
75%         43.000000
max        999.000000
Name: original_dti, dtype: float64

Lowest values:
original_dti
1       2
2       6
3       7
4       5
5      19
6      15
7      45
8      45
9      60
10     88
11    111


In [22]:
for col in [
    "credit_score",
    "original_dti",
    "number_of_borrowers",
]:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print(model_base[col].describe())

    print("\nLowest values:")
    print(model_base[col].value_counts().sort_index().head(15))

    print("\nHighest values:")
    print(model_base[col].value_counts().sort_index().tail(15))


credit_score
count    47255.000000
mean       749.152661
std         47.731212
min        462.000000
25%        717.000000
50%        759.000000
75%        788.000000
max        832.000000
Name: credit_score, dtype: float64

Lowest values:
credit_score
462    1
476    1
479    1
482    1
483    1
485    1
486    1
494    2
497    1
500    1
501    1
505    1
507    1
509    1
512    2
Name: count, dtype: int64

Highest values:
credit_score
816    216
817    134
818     52
819     90
820     81
821     10
822     15
823     63
824      3
825      7
826      3
827      2
829      5
831      1
832      2
Name: count, dtype: int64

original_dti
count    47255.000000
mean       113.518273
std        265.696033
min          1.000000
25%         28.000000
50%         36.000000
75%         43.000000
max        999.000000
Name: original_dti, dtype: float64

Lowest values:
original_dti
1       2
2       6
3       7
4       5
5      19
6      15
7      45
8      45
9      60
10     88
11    111


In [23]:
print(
    "DTI = 999:",
    (model_base["original_dti"] == 999).sum(),
    f"({(model_base['original_dti'] == 999).mean():.2%})",
)

print(
    "VantageScore = 9999:",
    (model_base["vantage_score_4"] == 9999).sum(),
    f"({(model_base['vantage_score_4'] == 9999).mean():.2%})",
)

print(
    "First-time homebuyer = 9:",
    (model_base["first_time_homebuyer_flag"] == "9").sum(),
)

DTI = 999: 3899 (8.25%)
VantageScore = 9999: 47255 (100.00%)
First-time homebuyer = 9: 1


In [24]:
for col in ["credit_score", "original_dti"]:
    print(f"\n{col}")
    print("-" * 50)

    print("Min:", model_base[col].min())
    print("Max:", model_base[col].max())

    print("\nLowest values:")
    print(model_base[col].value_counts().sort_index().head(15))

    print("\nHighest values:")
    print(model_base[col].value_counts().sort_index().tail(15))


credit_score
--------------------------------------------------
Min: 462
Max: 832

Lowest values:
credit_score
462    1
476    1
479    1
482    1
483    1
485    1
486    1
494    2
497    1
500    1
501    1
505    1
507    1
509    1
512    2
Name: count, dtype: int64

Highest values:
credit_score
816    216
817    134
818     52
819     90
820     81
821     10
822     15
823     63
824      3
825      7
826      3
827      2
829      5
831      1
832      2
Name: count, dtype: int64

original_dti
--------------------------------------------------
Min: 1
Max: 999

Lowest values:
original_dti
1       2
2       6
3       7
4       5
5      19
6      15
7      45
8      45
9      60
10     88
11    111
12    177
13    198
14    279
15    370
Name: count, dtype: int64

Highest values:
original_dti
37     1558
38     1615
39     1694
40     1693
41     1794
42     1773
43     1883
44     2030
45     1806
46      753
47      693
48      775
49      653
50      655
999    3899
Name: coun

In [25]:
collateral_features = [
    "original_ltv",
    "original_cltv",
    "mi_percentage",
    "property_type",
    "occupancy_status",
]

for col in collateral_features:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print("dtype:", model_base[col].dtype)
    print("missing:", model_base[col].isna().sum())
    print("unique:", model_base[col].nunique(dropna=False))

    print("\nValue counts:")
    print(model_base[col].value_counts(dropna=False).sort_index().to_string())



original_ltv
dtype: int64
missing: 0
unique: 173

Value counts:
original_ltv
5         1
6         2
7         4
8         4
9         5
10        6
11       10
12       12
13       16
14       21
15       22
16       34
17       42
18       34
19       35
20       41
21       51
22       69
23       61
24       66
25      102
26       76
27       81
28       97
29       99
30      108
31      128
32      126
33      137
34      158
35      167
36      139
37      182
38      169
39      194
40      209
41      179
42      203
43      250
44      210
45      245
46      247
47      287
48      283
49      326
50      423
51      285
52      370
53      368
54      384
55      391
56      445
57      417
58      518
59      465
60      773
61      367
62      471
63      505
64      542
65      653
66      578
67      652
68      695
69      684
70     1136
71      652
72      720
73      844
74      856
75     2840
76      543
77      691
78      803
79      861
80     9849
81       7

In [26]:
model_base[["original_ltv", "original_cltv", "mi_percentage"]].describe()

,original_ltv,original_cltv,mi_percentage
count,47255.000000,47255.000000,47255.000000
mean,73.658322,74.565908,6.119289
std,18.070077,18.580612,11.295345
min,5.000000,5.000000,0.000000
25%,65.000000,66.000000,0.000000
50%,78.000000,79.000000,0.000000
75%,84.000000,85.000000,0.000000
max,999.000000,999.000000,40.000000


In [27]:
print("LTV > 100:")
print(
    (model_base["original_ltv"] > 100).sum(),
    f"({(model_base['original_ltv'] > 100).mean():.2%})",
)

print("\nCLTV > 100:")
print(
    (model_base["original_cltv"] > 100).sum(),
    f"({(model_base['original_cltv'] > 100).mean():.2%})",
)

print("\nCLTV < LTV:")
print(
    (model_base["original_cltv"] < model_base["original_ltv"]).sum(),
)

print("\nCLTV = LTV:")
print(
    (model_base["original_cltv"] == model_base["original_ltv"]).sum(),
)

print("\nCLTV > LTV:")
print(
    (model_base["original_cltv"] > model_base["original_ltv"]).sum(),
)

LTV > 100:
463 (0.98%)

CLTV > 100:
678 (1.43%)

CLTV < LTV:
0

CLTV = LTV:
44457

CLTV > LTV:
2798


In [30]:
mi_by_ltv = pd.crosstab(
    model_base["original_ltv"],
    model_base["mi_percentage"].gt(0),
)

print(mi_by_ltv.loc[70:100])

mi_percentage  False  True 
original_ltv               
70              1136      0
71               651      1
72               720      0
73               844      0
74               856      0
75              2839      1
76               542      1
77               691      0
78               802      1
79               861      0
80              9846      3
81                47     25
82                38    133
83                53    213
84                50    274
85                48    871
86                44    143
87                42    243
88                28    308
89                43    309
90                55   2514
91                30     96
92                28    258
93                23    236
94                35    255
95                40   5268
96                25     11
97                32    161
98                22      6
99                19      9
100               27     11


In [31]:
loan_features = [
    "original_upb",
    "original_interest_rate",
    "original_loan_term",
    "loan_purpose",
    "channel",
    "amortization_type",
    "prepayment_penalty_flag",
]

for col in loan_features:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print("dtype:", model_base[col].dtype)
    print("missing:", model_base[col].isna().sum())
    print("unique:", model_base[col].nunique(dropna=False))

    print("\nMost common values:")
    print(model_base[col].value_counts(dropna=False).head(20))


original_upb
dtype: int64
missing: 0
unique: 655

Most common values:
original_upb
417000    1239
200000     533
100000     520
150000     407
180000     351
120000     341
300000     338
160000     333
140000     291
130000     280
170000     267
250000     267
128000     264
110000     262
190000     256
125000     242
280000     241
176000     238
220000     237
240000     232
Name: count, dtype: int64

original_interest_rate
dtype: float64
missing: 0
unique: 324

Most common values:
original_interest_rate
4.125    6563
4.250    5161
3.875    4935
3.750    4323
4.000    4231
4.375    2957
3.625    2641
4.500    1964
4.625    1845
3.500    1771
3.250    1362
4.750    1342
3.375    1274
3.125    1225
3.000    1018
3.990    1007
4.875     684
2.875     652
5.000     191
3.950     182
Name: count, dtype: int64

original_loan_term
dtype: int64
missing: 0
unique: 77

Most common values:
original_loan_term
360    34313
180     9286
240     2471
120      524
300      338
276       33
288  

In [32]:
model_base[
    [
        "original_upb",
        "original_interest_rate",
        "original_loan_term",
    ]
].describe()

,original_upb,original_interest_rate,original_loan_term
count,47255.000000,47255.000000,47255.000000
mean,221901.089832,3.974441,314.566586
std,118704.199071,0.455129,76.330162
min,10000.000000,2.375000,96.000000
25%,130000.000000,3.750000,240.000000
50%,199000.000000,4.000000,360.000000
75%,296000.000000,4.250000,360.000000
max,974000.000000,5.500000,366.000000


In [33]:
admin_features = [
    "property_state",
    "postal_code",
    "msa",
    "seller_name",
    "super_conforming_flag",
    "special_eligibility_program",
    "harp_indicator",
    "pre_harp_loan_id",
    "property_valuation_method",
    "interest_only_indicator",
    "first_payment_date",
    "maturity_date",
]

for col in admin_features:
    print(f"\n{'=' * 60}")
    print(col)
    print("=" * 60)

    print("dtype:", model_base[col].dtype)
    print("missing:", model_base[col].isna().sum())
    print("missing %:", f"{model_base[col].isna().mean():.2%}")
    print("unique:", model_base[col].nunique(dropna=False))

    print("\nMost common values:")
    print(model_base[col].value_counts(dropna=False).head(15))


property_state
dtype: str
missing: 0
missing %: 0.00%
unique: 54

Most common values:
property_state
CA    7297
TX    3236
FL    2807
IL    2182
MI    1692
CO    1613
WA    1502
OH    1495
NC    1460
NY    1460
GA    1401
PA    1369
AZ    1280
VA    1256
MN    1186
Name: count, dtype: int64

postal_code
dtype: string
missing: 0
missing %: 0.00%
unique: 866

Most common values:
postal_code
945    623
750    617
300    457
852    406
606    390
980    385
840    383
600    378
917    370
926    367
601    355
802    346
913    338
956    320
928    312
Name: count, dtype: int64[pyarrow]

msa
dtype: string
missing: 5433
missing %: 11.50%
unique: 446

Most common values:
msa
<NA>     5433
31084    1532
19740     962
38060     945
12060     933
19124     905
26420     901
16984     865
47894     834
33460     815
40140     790
35614     718
42644     716
11244     716
38900     701
Name: count, dtype: int64[pyarrow]

seller_name
dtype: str
missing: 0
missing %: 0.00%
unique: 25

Most commo

In [34]:
eda = model_base.copy()

eda["dti_missing"] = eda["original_dti"].eq(999)

dti_missing_summary = eda.groupby("dti_missing")["ever_90dpd_24m"].agg(
    loans="size",
    events="sum",
    event_rate="mean",
)

dti_missing_summary["event_rate_pct"] = (dti_missing_summary["event_rate"] * 100).round(
    3
)

display(dti_missing_summary)

,loans,events,event_rate,event_rate_pct
dti_missing,,,,
False,43356,275,0.006343,0.634
True,3899,68,0.017440,1.744


In [35]:
model_base["original_dti"].median()

np.float64(36.0)

In [36]:
from credit_risk.features.origination import (
    select_baseline_features,
    normalize_sentinel_values,
    add_missing_indicators,
)

X = select_baseline_features(model_base)
X = normalize_sentinel_values(X)
X = add_missing_indicators(X)

missing_summary = (
    X.isna()
    .sum()
    .to_frame("missing")
    .assign(missing_pct=lambda x: (x["missing"] / len(X) * 100).round(2))
    .query("missing > 0")
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary)

,missing,missing_pct
original_dti,3899,8.25
first_time_homebuyer_flag,1,0.00
original_ltv,1,0.00
original_cltv,1,0.00


In [37]:
dates = pd.to_datetime(
    model_base["first_payment_date"],
    format="%Y%m",
)

print("Earliest:", dates.min())
print("Latest:", dates.max())

display(dates.dt.to_period("M").value_counts().sort_index())

Earliest: 2015-02-01 00:00:00
Latest: 2017-05-01 00:00:00


first_payment_date
2015-02      37
2015-03    2863
2015-04    3833
2015-05    5003
2015-06    4144
2015-07    3922
2015-08    3781
2015-09    3961
2015-10    3902
2015-11    3896
2015-12    4262
2016-01    3517
2016-02    3951
2016-03     123
2016-04      10
2016-05       4
2016-06       5
2016-07       4
2016-08       5
2016-09      12
2016-10       6
2016-11       4
2016-12       4
2017-01       2
2017-02       2
2017-04       1
2017-05       1
Freq: M, Name: count, dtype: int64

In [38]:
tmp = model_base.assign(first_payment_month=dates.dt.to_period("M"))

monthly_target = tmp.groupby("first_payment_month")["ever_90dpd_24m"].agg(
    loans="size",
    events="sum",
    event_rate="mean",
)

monthly_target["event_rate_pct"] = (monthly_target["event_rate"] * 100).round(3)

display(monthly_target)

,loans,events,event_rate,event_rate_pct
first_payment_month,,,,
2015-02,37,0,0.000000,0.000
2015-03,2863,19,0.006636,0.664
2015-04,3833,24,0.006261,0.626
2015-05,5003,24,0.004797,0.480
2015-06,4144,24,0.005792,0.579
2015-07,3922,21,0.005354,0.535
2015-08,3781,27,0.007141,0.714
2015-09,3961,33,0.008331,0.833
2015-10,3902,29,0.007432,0.743
